In [1]:
import numpy as np

In [4]:
import pandas as pd

In [6]:
!pip install yfinance

In [7]:
!pip install pytrends

In [8]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu


In [9]:
!pip install d3rlpy

In [11]:
!pip install xgboost

In [1]:
!pip install gymnasium

In [4]:
from Data_handler import MarketDataHandler, GoogleTrendHandler

In [6]:
from Integrate_class import Integrated

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [7]:
from backtest import run_per_ticker, compute_metrics

In [34]:

import numpy as np
import pandas as pd

from Data_handler import MarketDataHandler, GoogleTrendHandler
from Integrate_class import Integrated
from backtest import run_per_ticker, compute_metrics


def main():
    # ========== 1. 配置 ==========
    USE_FAKE_DATA = False  # True 时用假数据快速测试，False 用雅虎+Google Trends

    tickers = ["AAPL", "MSFT", "GOOGL"]
    keywords = ["Apple stock", "Microsoft stock", "Google stock"]

    start_date = "2020-01-01"
    end_date = "2025-01-01"

    # ========== 2. 准备数据：returns / trends / prices_bt ==========
    if USE_FAKE_DATA:
        dates = pd.date_range(start="2020-01-01", periods=200, freq="B")
        rng = np.random.default_rng(0)

        returns = pd.DataFrame(
            rng.normal(0, 0.01, size=(len(dates), len(tickers))),
            index=dates,
            columns=tickers,
        )

        trend_df = pd.DataFrame(
            rng.integers(0, 100, size=(len(dates), len(keywords))),
            index=dates,
            columns=keywords,
        )

        prices_bt = (1 + returns).cumprod()

    else:
        # 真实市场数据
        market = MarketDataHandler(tickers, start_date, end_date)
        prices = market.load_data()
        clean_prices = market.clean_data()
        returns = market.compute_returns()      # 同时会保存 returns_data.xlsx

        # Google Trends
        gt = GoogleTrendHandler(keywords, start_date=start_date, end_date=end_date)
        trend_df = gt.load_trends()            # 同时会保存 googletrend_data.xlsx

        # 回测用的价格：clean_prices
        prices_bt = clean_prices

    # ========== 3. 调用 Integrated 训练所有模型（多资产多输出） ==========
    dudu = Integrated()

    model_map = dudu.return_trend(
        returns=returns,
        trends=trend_df,
        tickers=tickers,
        client_text="",
    )


    print("\n===== 训练出的模型类别 =====")
    print("Pred models:", [name for name, _ in model_map["pred"]])
    print("Cls  models:", [name for name, _ in model_map["cls"]])
    print("RL   models:", [name for name, _ in model_map["rl"]])

    # ========== 4. 把模型分配给每只股票 ==========

    all_models = (
        model_map["pred"]
        + model_map["cls"]
        + model_map["rl"]
    )

    models_for_bt = {tkr: all_models for tkr in tickers}

    # ========== 5. 运行回测 ==========
    table, equities, weights = run_per_ticker(
        prices=prices_bt,
        models=models_for_bt,
        strategy="long_short",
        thr_long=0.0,
        thr_short=None,
        hold=10,
        fee_buy=0.0005,
        fee_sell=0.0005,
        leverage_cap=1.0,
        state_window=60,
        warmup_days=0,
        feature_maker=None,     # 使用 backtest.py 里的 default_feature_maker
    )

    # ========== 6. 汇总绩效指标 ==========
    rows = []
    for (tkr, name), eq in equities.items():
        m = compute_metrics(eq)
        rows.append(
            (tkr, name, m["CAGR"], m["Sharpe"], m["MaxDD"], m["AnnRet"], m["AnnVol"])
        )

    perf_df = pd.DataFrame(
        rows,
        columns=["Ticker", "Model", "CAGR", "Sharpe", "MaxDD", "AnnRet", "AnnVol"],
    ).set_index(["Ticker", "Model"])
    
    return perf_df, equities


if __name__ == "__main__":
    perf_df, equities = main()
   
    print("\n===== Backtest Performance =====\n")
    print(perf_df.round(3))

    # ========== 7. 调用 LLM 对回测结果做点评 ==========
    try:
        from backtest_llm import BacktestLLMAnalyst

        analyst = BacktestLLMAnalyst(
            api_key="你的_API_Key_或者留空用环境变量",
            # base_url / model 如有需要可在这里改
            verbose=False,
        )

        comment = analyst.analyze(
            perf_df=perf_df,
            equities=equities,
            params={
                "strategy": "long_short",
                "hold": 10,
                "fee_buy": 0.0005,
                "fee_sell": 0.0005,
                "state_window": 60,
            },
            language="zh",   # 想要英文就改成 "en"
        )

        print("\n===== LLM Analysis of Backtest =====\n")
        print(comment)

    except Exception as e:
        print("\n[WARN] LLM 分析失败，仅打印错误信息：", e)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
[*********************100%***********************]  3 of 3 completed


Loaded 1258 days of data for 3 tickers.
Cleaned data — shape: (1258, 3)
Saved returns_data.xlsx


/Users/doris/Desktop/4511(4)/4511/Data_handler.py:159: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.trends = self.trends.fillna(method='ffill')


Google Trends data saved to googletrend_data.xlsx
=== MERGE DIAGNOSTICS (multi-asset) ===
returns:  2020-01-03 00:00:00 → 2024-12-27 00:00:00  rows=1255  cols=3
trends:   2020-01-03 00:00:00 → 2024-12-27 00:00:00  rows=1301  cols=3
overlap:  2020-01-03 00:00:00 → 2024-12-27 00:00:00  rows=1255
return_cols: ['AAPL', 'GOOGL', 'MSFT']
trend_cols:  ['Apple stock', 'Microsoft stock', 'Google stock']
feature_cols: ['AAPL_ret_mom_5d', 'AAPL_ret_vol_5d', 'GOOGL_ret_mom_5d', 'GOOGL_ret_vol_5d', 'MSFT_ret_mom_5d', 'MSFT_ret_vol_5d', 'Apple stock_trend_mom_5d', 'Microsoft stock_trend_mom_5d', 'Google stock_trend_mom_5d']
Merged dataset saved to merged_features_multiasset.xlsx
=== FINAL DATA SHAPE ===
X shape: (1249, 12)
y shape: (1249, 3)
2025-11-21 02:12.38 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('int64')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(12,)]) reward_signature=Signature(dtype=[dtype('floa

Epoch 1/3:   0%|          | 0/1000 [00:00<?, ?it/s]

...

Epoch 3/3:   0%|          | 0/1000 [00:00<?, ?it/s]

2025-11-21 02:12.43 [info     ] DiscreteCQL_20251121021238: epoch=3 step=3000 epoch=3 metrics={'time_sample_batch': 0.0009769744873046875, 'time_algorithm_update': 0.0009395904541015625, 'loss': 0.5593991220593453, 'td_loss': 0.0205040410682559, 'conservative_loss': 0.5388950819075108, 'time_step': 0.0019320712089538575} step=3000
2025-11-21 02:12.43 [info     ] Model parameters are saved to d3rlpy_logs/DiscreteCQL_20251121021238/model_3000.d3
DEBUG build_cls_labels_from_returns:
  y_raw sample: [ 1.37118880e-02 -9.03396865e-03  2.73651863e-03  1.28216479e-02
  1.23045688e-02 -2.84071061e-03 -4.76926725e-05  3.84165444e-03
 -8.47754040e-03 -2.32061535e-02]
  labels sample: [1 0 1 1 1 0 0 1 0 0]
  unique labels: [0 1]

===== 训练出的模型类别 =====
Pred models: ['LSTM_0', 'LSTM_1', 'LSTM_2', 'LinearRegression', 'RandomForestRegressor', 'BayesianRidge']
Cls  models: ['XGBClassifier', 'RandomForestClassifier']
RL   models: ['CQL_policy']


/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(



===== Backtest Performance =====

                                CAGR  Sharpe  MaxDD  AnnRet  AnnVol
Ticker Model                                                       
AAPL   LSTM_0                  0.139   0.590 -0.477   0.173   0.294
       LSTM_1                  0.137   0.583 -0.531   0.172   0.296
       LSTM_2                  0.094   0.448 -0.316   0.135   0.302
       LinearRegression        0.230   0.835 -0.271   0.253   0.304
       RandomForestRegressor   0.171   0.695 -0.413   0.200   0.288
       BayesianRidge           0.291   1.003 -0.335   0.301   0.300
       XGBClassifier           0.281   0.942 -0.314   0.298   0.317
       RandomForestClassifier  0.281   0.942 -0.314   0.298   0.317
       CQL_policy              0.090   0.441 -0.402   0.130   0.295
MSFT   LSTM_0                  0.056   0.335 -0.424   0.094   0.282
       LSTM_1                  0.072   0.389 -0.468   0.109   0.281
       LSTM_2                 -0.054  -0.047 -0.551  -0.014   0.290
       Linear

/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


In [36]:
    print("\n===== Backtest Performance =====\n")
    print(perf_df.round(3))

    # ========== 7. 调用 LLM 对回测结果做点评 ==========
    try:
        from backtest_llm import BacktestLLMAnalyst

        analyst = BacktestLLMAnalyst()  # 可以在里面配置 deepseek / openai 等
        comment = analyst.analyze(
            perf_df=perf_df,
            equities=equities,
            params={
                "strategy": "long_short",
                "hold": 10,
                "fee_buy": 0.0005,
                "fee_sell": 0.0005,
                "state_window": 60,
            }
        )
        print("\n===== LLM Analysis of Backtest =====\n")
        print(comment)
    except Exception as e:
        print("\n[WARN] LLM 分析失败，仅打印错误信息：", e)



===== Backtest Performance =====

                                CAGR  Sharpe  MaxDD  AnnRet  AnnVol
Ticker Model                                                       
AAPL   LSTM_0                  0.139   0.590 -0.477   0.173   0.294
       LSTM_1                  0.137   0.583 -0.531   0.172   0.296
       LSTM_2                  0.094   0.448 -0.316   0.135   0.302
       LinearRegression        0.230   0.835 -0.271   0.253   0.304
       RandomForestRegressor   0.171   0.695 -0.413   0.200   0.288
       BayesianRidge           0.291   1.003 -0.335   0.301   0.300
       XGBClassifier           0.281   0.942 -0.314   0.298   0.317
       RandomForestClassifier  0.281   0.942 -0.314   0.298   0.317
       CQL_policy              0.090   0.441 -0.402   0.130   0.295
MSFT   LSTM_0                  0.056   0.335 -0.424   0.094   0.282
       LSTM_1                  0.072   0.389 -0.468   0.109   0.281
       LSTM_2                 -0.054  -0.047 -0.551  -0.014   0.290
       Linear